# DeepLOB

Install `requirements.txt`, select that Python environment as the notebook kernel, and run all sections in order from the project root. This notebook loads the existing prepared daily arrays and creates the chronological datasets, DataLoaders, and DeepLOB model. Training and test evaluation run only when their switches are enabled.

Data extraction, normalization, and label generation are handled separately by `prepare_data.py`; see `README.md` for the processing rules and commands.


## 1. Environment

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

display(pd.Series({
    "Python": sys.version.split()[0],
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "PyTorch": torch.__version__,
}, name="Version"))
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

## 2. Configuration

Select the prepared feature directory and labels for the desired horizon. The current labels use `k=20`, `alpha=0.0000263`, a past/current mean over `k+1` prices, and a future mean over `k` prices. Changing the threshold requires generating labels with `prepare_data.py` before loading them here.


In [ ]:
PROJECT_DIR = Path(".")  # Run the notebook from the project root.
EXPERIMENT = "experiment_03"  # BatchNorm, He normal, and lower learning rate.
STOCK = "KGHM"
WINDOW_LENGTH = 50
N_LEVELS = 10
BATCH_NORM = True
DROPOUT = 0.2  # Shared channel mask across time; disabled during evaluation.
INITIALIZATION = "he_normal"  # Convolutions; LSTM recurrent weights are orthogonal.
HORIZON = 20
LABEL_ALPHA = 0.0000263  # Describes the stored labels; changing it does not regenerate them.
BATCH_SIZE = 64
NUM_WORKERS = 0
LOADER_SEED = 42
TRAINING_SEED = 42
LEARNING_RATE = 0.0001
ADAM_EPS = 1e-8  # PyTorch Adam default.
MAX_EPOCHS = 50
PATIENCE = 20
SELECTION_METRIC = "accuracy"
DEVICE = "cuda"  # CUDA GPU; use "mps" for an Apple GPU.
RUN_TRAINING = False
RUN_TESTING = False
SPLIT_DATES = {
    "train": ("20170101", "20170831"),
    "validation": ("20170901", "20170930"),
    "test": ("20171001", "20171231"),
}

PREPARED_DIR = PROJECT_DIR / "data" / "wselob_2017_prepared" / STOCK
FEATURE_DIR = PREPARED_DIR / "features"
LABEL_DIR = PREPARED_DIR / "labels" / f"k{HORIZON}"
RUN_DIR = PROJECT_DIR / "experiments" / EXPERIMENT / STOCK

display(pd.Series({
    "Experiment": EXPERIMENT,
    "Stock": STOCK,
    "BatchNorm": BATCH_NORM,
    "Dropout": DROPOUT,
    "Initialization": INITIALIZATION,
    "Window length": WINDOW_LENGTH,
    "Levels per side": N_LEVELS,
    "Prediction horizon": HORIZON,
    "Stored label threshold": LABEL_ALPHA,
    "Batch size": BATCH_SIZE,
    "Loader workers": NUM_WORKERS,
    "Loader seed": LOADER_SEED,
    "Training seed": TRAINING_SEED,
    "Adam learning rate": LEARNING_RATE,
    "Adam epsilon": ADAM_EPS,
    "Maximum epochs": MAX_EPOCHS,
    "Early-stopping patience": PATIENCE,
    "Checkpoint metric": SELECTION_METRIC,
    "Requested training device": DEVICE,
    "Run training": RUN_TRAINING,
    "Run test evaluation": RUN_TESTING,
    "Run directory": str(RUN_DIR),
    "Feature directory": str(FEATURE_DIR),
    "Label directory": str(LABEL_DIR),
}, name="Configuration"))

## 3. Define input windows

`DailyWindowDataset` takes lists of daily feature arrays and integer label arrays, with one label position per feature row. It returns a feature window and the label at its last row. The following section loads the prepared arrays and creates one Dataset for each chronological split.

In [ ]:
from train_deeplob import DailyWindowDataset


## 4. Load the chronological datasets

Assign complete trading dates using `SPLIT_DATES`. The first history-only days have no prepared files and therefore contribute no samples. Read each selected feature and label file once, preserving daily boundaries; the Dataset slices overlapping windows when requested.

In [ ]:
feature_paths = sorted(FEATURE_DIR.glob("*.npz"))
split_files = {}
split_datasets = {}

for split_name, (start_date, end_date) in SPLIT_DATES.items():
    # Select whole days; filenames use their Europe/Warsaw trading dates.
    paths = []
    for path in feature_paths:
        if start_date <= path.stem <= end_date:
            paths.append(path)
    split_files[split_name] = paths

    daily_features = []
    daily_labels = []
    for path in paths:
        with np.load(path, allow_pickle=False) as arrays:
            daily_features.append(arrays["features"])
        with np.load(LABEL_DIR / path.name, allow_pickle=False) as arrays:
            daily_labels.append(arrays["labels"])

    split_datasets[split_name] = DailyWindowDataset(
        daily_features, daily_labels,
        window_length=WINDOW_LENGTH, horizon=HORIZON,
    )
    print(f"{split_name}: {len(paths)} days, {len(split_datasets[split_name]):,} samples")

train_dataset = split_datasets["train"]
validation_dataset = split_datasets["validation"]
test_dataset = split_datasets["test"]

## 5. Create the DataLoaders

Shuffle training windows with a dedicated seeded generator. Validation and test windows remain chronological. Use one process and keep the final partial batch in every split. Batches contain feature tensors of shape `(batch, window_length, features)` and one integer target per sample.

In [ ]:
train_generator = torch.Generator().manual_seed(LOADER_SEED)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=False, generator=train_generator,
)
validation_loader = torch.utils.data.DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, drop_last=False,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, drop_last=False,
)

## 6. Create the DeepLOB model

Use the architecture in DeepLOB Section IV and Figures 3-4: convolution channels 16, three Inception branches with 32 channels each, and one LSTM with 64 hidden units. Temporal padding preserves the input window length. Convolution activations use LeakyReLU with slope 0.01. The baseline has no BatchNorm or dropout.

`deeplob.py` accepts `(batch, time, 4 * n_levels)` and returns three logits for the final prediction position. Pass these logits directly to `CrossEntropyLoss`; apply softmax only when probabilities are needed. The current input window has 50 time steps rather than the paper's 100. Experiment 3 adds BatchNorm after all 14 convolution activations and uses He normal convolution weights. LSTM input weights and the classifier use Xavier uniform; the stacked LSTM recurrent matrix is orthogonal, with total forget-gate bias one. Dropout with probability 0.2 is applied after Inception and before the LSTM. It is disabled during validation and testing.

This section creates the model on CPU for inspection. The training section initializes a fresh model with the same seed and moves it to `DEVICE`.

In [ ]:
import importlib
import deeplob

# Reload model edits when rerunning this notebook in the same kernel.
importlib.reload(deeplob)
random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
model = deeplob.DeepLOB(
        n_levels=N_LEVELS, batch_norm=BATCH_NORM,
        dropout=DROPOUT, initialization=INITIALIZATION,
    )
criterion = torch.nn.CrossEntropyLoss()

parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"DeepLOB trainable parameters: {parameter_count:,}")

## 7. Train DeepLOB

Use Adam with learning rate 0.0001, epsilon 1e-8, betas (0.9, 0.999), and no weight decay. Use batch size 64 and at most 50 epochs. Stop after 20 epochs without a strictly higher validation accuracy. Record loss, accuracy, and F1 weighted by true class support, matching OFN's F1 aggregation.

Enable `RUN_TRAINING` after the experiment configuration is confirmed and the GPU environment is ready. Training starts from fresh weights and writes `best_model.pt` and `history.csv` under a new `RUN_DIR`. Existing run directories are rejected. After training ends, save loss, accuracy, and weighted F1 curves in `training_curves.png`.


In [ ]:
import train_deeplob

importlib.reload(train_deeplob)

if RUN_TRAINING:
    random.seed(TRAINING_SEED)
    np.random.seed(TRAINING_SEED)
    torch.manual_seed(TRAINING_SEED)
    train_loader.generator.manual_seed(LOADER_SEED)
    device = torch.device(DEVICE)
    model = deeplob.DeepLOB(
        n_levels=N_LEVELS, batch_norm=BATCH_NORM,
        dropout=DROPOUT, initialization=INITIALIZATION,
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPS,
        betas=(0.9, 0.999), weight_decay=0,
    )

    # Save the data scope and settings with the selected model, not in daily NPZ files.
    data_dates = {}
    for split, paths in split_files.items():
        data_dates[split] = [path.stem for path in paths]
    run_config = {
        "stock": STOCK, "window_length": WINDOW_LENGTH, "n_levels": N_LEVELS,
        "batch_norm": BATCH_NORM, "dropout": DROPOUT, "initialization": INITIALIZATION,
        "horizon": HORIZON, "label_alpha": LABEL_ALPHA,
        "label_formula": "past=sum(p[t-k:t+1])/(k+1); future=sum(p[t+1:t+k+1])/k; change=(future-past)/past",
        "feature_dir": str(FEATURE_DIR), "label_dir": str(LABEL_DIR),
        "data_dates": data_dates, "class_names": ["down", "stationary", "up"],
        "sample_counts": {
            "train": len(train_loader.dataset), "validation": len(validation_loader.dataset),
            "test": len(test_loader.dataset),
        },
        "training_seed": TRAINING_SEED, "loader_seed": LOADER_SEED,
        "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "drop_last": False,
        "optimizer": "Adam", "learning_rate": LEARNING_RATE, "eps": ADAM_EPS,
        "betas": [0.9, 0.999], "weight_decay": 0,
        "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
        "selection_metric": SELECTION_METRIC, "f1_average": "weighted", "device": str(device),
        "python": sys.version.split()[0], "numpy": np.__version__,
        "pytorch": str(torch.__version__),
    }
    history = train_deeplob.train_model(
        model, train_loader, validation_loader, optimizer, criterion, device,
        RUN_DIR, epochs=MAX_EPOCHS, patience=PATIENCE,
        selection_metric=SELECTION_METRIC, config=run_config,
    )
    display(pd.DataFrame(history).tail())
    curve_path = train_deeplob.plot_training_history(history, RUN_DIR)
    display(Image(filename=str(curve_path)))
else:
    print("Training is disabled. Enable RUN_TRAINING and select a new RUN_DIR for a GPU run.")

## 8. Evaluate the selected checkpoint

Enable `RUN_TESTING` after training has finished. This cell reloads `best_model.pt` and evaluates the test split in chronological order, without updating weights. It reports accuracy, weighted F1, per-class precision/recall/F1, and a confusion matrix (true classes in rows, predictions in columns).

The baseline always predicts the most frequent class among eligible **training** targets. Both the model and baseline are scored on the same test labels. Classes with no predictions receive precision and F1 of zero. Results are saved in `test_results.txt`; existing reports are not overwritten. Save `confusion_matrix.png` with counts and row percentages, and `test_metrics.png` with baseline and per-class comparisons.

The same aggregation is used for validation and testing. Counts are accumulated batch by batch, so predictions for the entire year do not need to be retained in memory. Mean loss is weighted by sample count, including the final partial batch.

In [ ]:
if RUN_TESTING:
    device = torch.device(DEVICE)
    checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location=device, weights_only=True)
    model = deeplob.DeepLOB(
        n_levels=checkpoint["config"]["n_levels"],
        batch_norm=checkpoint["config"].get("batch_norm", False),
        dropout=checkpoint["config"].get("dropout", 0.0),
        initialization=checkpoint["config"].get("initialization", "pytorch"),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])

    # Open the new report before the expensive test pass; a rerun cannot overwrite it.
    with (RUN_DIR / "test_results.txt").open("x") as report:
        test_metrics = train_deeplob.run_epoch(model, test_loader, criterion, device)
        training_counts = np.zeros(3, dtype=np.int64)
        for labels in train_dataset.daily_labels:
            targets = labels[train_dataset.first_position:len(labels) - HORIZON]
            training_counts += np.bincount(targets, minlength=3)
        majority_class = int(training_counts.argmax())
        baseline_confusion = np.zeros((3, 3), dtype=np.int64)
        baseline_confusion[:, majority_class] = test_metrics["support"]
        baseline_metrics = train_deeplob.classification_metrics(baseline_confusion)

        summary = pd.DataFrame([
            {"Model": "DeepLOB", "Accuracy": test_metrics["accuracy"],
             "Weighted F1": test_metrics["weighted_f1"]},
            {"Model": "Training-majority baseline", "Accuracy": baseline_metrics["accuracy"],
             "Weighted F1": baseline_metrics["weighted_f1"]},
        ]).set_index("Model")
        class_names = ["down", "stationary", "up"]
        per_class = pd.DataFrame({
            "Precision": test_metrics["precision"], "Recall": test_metrics["recall"],
            "F1": test_metrics["f1"], "Support": test_metrics["support"],
        }, index=class_names)
        confusion = pd.DataFrame(test_metrics["confusion"], index=class_names, columns=class_names)
        confusion.index.name = "True class"
        confusion.columns.name = "Predicted class"

        print(f"Checkpoint epoch: {checkpoint['epoch']}; test loss: {test_metrics['loss']:.6f}")
        display(summary, per_class, confusion)
        report.write(f"Checkpoint epoch: {checkpoint['epoch']}\n")
        report.write(f"Selection: validation {checkpoint['selection_metric']} = {checkpoint['best_score']:.6f}\n")
        report.write(f"Test loss: {test_metrics['loss']:.6f}\n")
        report.write(f"Training-majority class: {majority_class} ({class_names[majority_class]})\n\n")
        report.write(summary.to_string() + "\n\n" + per_class.to_string() + "\n\n")
        report.write(confusion.to_string() + "\n")

    # Plot the accumulated test counts; no second test pass is needed.
    confusion_path = train_deeplob.plot_confusion_matrix(test_metrics, class_names, RUN_DIR)
    metrics_path = train_deeplob.plot_test_metrics(test_metrics, baseline_metrics, class_names, RUN_DIR)
    display(Image(filename=str(confusion_path)), Image(filename=str(metrics_path)))
else:
    print("Test evaluation is disabled. Enable RUN_TESTING after selecting the validation checkpoint.")